# Injury severity: gamma GLM

**What it predicts:** the number of days a soccer player misses after an injury.

**Model:** gamma GLM with a log link, fit on completed injuries (1 to 1,000 days missed).

```
log E[days missed] = severity tier + position + age + age^2
                     + log(1 + prior injuries) + re-injury within 60 days + prior injury to the same body part
```

Severity tiers group injury rating classes (body region x injury type) with similar absence lengths. Position is compared against midfielders; age is centred at 26.

**Headline result (held-out players):** mean absolute error of **33.3 days**, against **44.0 days** for a model with no predictors.

In [1]:
import numpy as np
import pandas as pd
import severity as sev

data = sev.build_dataset()
print(f"{len(data):,} injuries, {data.pid.nunique():,} players, {data.injury_class.nunique()} rating classes")

163,414 injuries, 34,429 players, 57 rating classes


## Held-out evaluation

Players are split 80/20 (each player's injuries all land on one side). Both models are fit on the training players and scored on the completed injuries of the test players. Lower `mae`, `rmse` and `gamma_dev` are better; `above_p90` should be close to 0.10 if the 90th-percentile prediction is well calibrated.

In [2]:
train, test = sev.split(data)
sev.compare(sev.fit_all(train), test)

  fitting Gamma, no predictors ...


  fitting Gamma GLM, severity tiers ...


,log_lik,gamma_dev,mae,rmse,mean_pred,above_p90
model,,,,,,
"Gamma, no predictors",-4.8515,1.3912,44.02,70.97,48.24,0.1055
"Gamma GLM, severity tiers",-4.6184,0.9286,33.27,59.32,48.27,0.0894
(actual),NaN,NaN,NaN,NaN,47.51,NaN


## Final fit

The model used for the coefficients and tiers below is refit on all injuries.

In [3]:
glm = sev.TieredGammaGLM().fit(data)

## Coefficients

**How to read:** `multiplier` = exp(coef) is the factor applied to expected days missed. For example, 1.40 means 40% longer absences and 0.80 means 20% shorter, holding the other predictors fixed. The reference is the tier containing Hamstring (unspecified) and a midfielder (Tier 8 in the current fit); the intercept's multiplier is the expected days for that reference case at age 26 with no injury history.

In [4]:
coefs = sev.coefficients(glm)
coefs.insert(1, "multiplier", np.exp(coefs["coef"]).round(3))
coefs

,coef,multiplier,p_value
variable,,,
Intercept,3.9558,52.237,0.0000
severity_tier = Tier 1,-1.3462,0.260,0.0000
severity_tier = Tier 2,-0.7536,0.471,0.0000
severity_tier = Tier 3,-0.6301,0.533,0.0000
severity_tier = Tier 4,-0.4838,0.616,0.0000
severity_tier = Tier 5,-0.2895,0.749,0.0000
severity_tier = Tier 6,-0.1937,0.824,0.0000
severity_tier = Tier 7,-0.1051,0.900,0.0000
severity_tier = Tier 9,0.1999,1.221,0.0000


## Severity tiers

Each tier is a set of rating classes merged until neighbouring tiers differ at p < 0.05. Tier 1 has the shortest absences.

In [5]:
tier_of = pd.Series(glm.tier_of)
tiers = tier_of.groupby(tier_of).apply(lambda s: "; ".join(sorted(s.index)))
tiers = tiers.reindex(glm.tier_names).rename("rating classes").to_frame()
tiers.index.name = "tier"
with pd.option_context("display.max_colwidth", None):
    display(tiers)

,rating classes
tier,
Tier 1,Illness (unspecified)
Tier 2,Head - Concussion; Knee - Knock / bruise; Trunk - Knock / bruise; Unknown - Knock / bruise; Unknown - Strain / muscle
Tier 3,Foot - Knock / bruise; Knock / bruise - other sites
Tier 4,Ankle - Ligament / joint; Head (unspecified); Muscle (unspecified) - Strain / muscle; Strain / muscle - other sites; Thigh (unspecified); Thigh - Knock / bruise
Tier 5,Calf (unspecified); Calf - Strain / muscle; Hamstring - Strain / muscle; Head - Fracture
Tier 6,Groin (unspecified); Muscle (unspecified) - Tear / rupture; Unknown (unspecified)
Tier 7,Arm / hand (unspecified); Back / neck (unspecified); Trunk - Fracture
Tier 8,Ankle (unspecified); Arm / hand - Fracture; Calf - Tear / rupture; Foot (unspecified); Groin - Tear / rupture; Hamstring (unspecified); Other (rare); Trunk - Strain / muscle; Unknown - Tendon / inflammation
Tier 9,Achilles - Tendon / inflammation; Hip (unspecified); Unspecified injury - other sites


## More

- [MODEL_SUMMARY.md](MODEL_SUMMARY.md): plain-language summary of both models, with worked examples.
- [README.md](README.md): setup, data sources, and how to run the scripts.
- Code: [severity.py](severity.py) and [career.py](career.py).